In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
# Crear widgets
dbutils.widgets.text("catalogo", "catalog_cr")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
# Obtener valores de widgets
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
# Crear Dataframes a partir de tablas crime_reports, crimes y months.
df_crime_reports = spark.table(f"{catalogo}.{esquema_source}.crime_reports")
df_crimes = spark.table(f"{catalogo}.{esquema_source}.crimes")
df_months = spark.table(f"{catalogo}.{esquema_source}.months")

In [0]:
# Contar filas de df_crime_reports
df_crime_reports.count()

In [0]:
# Filtrar registros del 2021 en adelante
new_df_crime_reports = df_crime_reports.filter(col("report_year") >= 2021)

In [0]:
# Eliminar columna emergency_district (debido a que solo abarca Lima Metropolitana)
new_df_crime_reports = new_df_crime_reports.drop("emergency_district") 
new_df_crime_reports.display()

In [0]:
# Contar filas de new_df_crime_reports
new_df_crime_reports.count()

In [0]:
# Unir los Dataframes df_crime_reports y df_crimes por la columna crime_id
df_crime_reports_joined = new_df_crime_reports.join(df_crimes, "crime_id", how="inner")
df_crime_reports_joined.display()

In [0]:
# Unir los Dataframes df_crime_reports_joined y df_months por la columna month_id
df_crime_reports_joined = df_crime_reports_joined.join(df_months, "month_id", how="inner")
df_crime_reports_joined.display()

In [0]:
# Contar filas de df_crime_reports_joined
df_crime_reports_joined.count()

In [0]:
# Borrar columnas ingestion_date
df_crime_reports_joined = df_crime_reports_joined.drop("ingestion_date")

In [0]:
# Limpiar data de valores nulos
df_crime_reports_cleaned = df_crime_reports_joined.dropna(how="any")
df_crime_reports_cleaned.display()

In [0]:
# Contar filas de df_crime_reports_cleaned
df_crime_reports_cleaned.count()

In [0]:
# Borrar columnas month_id y crime_id
df_crime_reports_cleaned = df_crime_reports_cleaned.drop("month_id", "crime_id")
df_crime_reports_cleaned.display()

In [0]:
# Contar años únicos
df_crime_reports_cleaned.select("report_year").distinct().count()

In [0]:
# Agrupar por ubigeo, tipo de crimen y año
df_crime_reports_filtered  = df_crime_reports_cleaned.groupBy("ubigeo","department", "province", "district","crime", "report_year") \
    .agg(
        sum("n_crimes").alias("total_n_crimes")
    )

df_crime_reports_filtered.display()

In [0]:
# Contar filas de df_crime_reports_filtered
df_crime_reports_filtered.count()

In [0]:
# Ordenar de manera alfabética
df_crime_reports_filtered = df_crime_reports_filtered.sort(asc("department"), asc("province"), asc("district"), asc("crime"), asc("report_year"))
df_crime_reports_filtered.display()


In [0]:
# Añadir columna id_report a df_crime_reports_filtered desde el n°1
df_crime_reports_filtered = df_crime_reports_filtered.withColumn("report_id", monotonically_increasing_id() + 1)
df_crime_reports_filtered.display()


In [0]:
# Seleccionar columnas específicas, con report_id al inicio. Agregar columna ingestion_date.
df_crime_reports_reordered = df_crime_reports_filtered.select(
    col("report_id"),
    col("ubigeo"),
    col("department"),
    col("province"),
    col("district"),
    col("crime").alias("crime_name"),
    col("report_year"),
    col("total_n_crimes")
).withColumn("ingestion_date", current_timestamp())
df_crime_reports_reordered.display()

In [0]:
# Contar filas de df_crime_reports_reordered
df_crime_reports_reordered.count()

In [0]:
# Ingestar data en la tabla "crime_reports_transformed"
df_crime_reports_reordered.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.crime_reports_transformed")